In [1]:
# First we got to do some imports
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import time

from sklearn.linear_model import LinearRegression


from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline

import contextlib, io
from itertools import combinations
import shap


# Random seed for reproducibility
np.random.seed(42)

all_targetv = ["tradeflow_baci", "tradeflow_comtrade_o", "tradeflow_comtrade_d", 
                "tradeflow_imf_o", "tradeflow_imf_d", "combined_trade_baci",]

df = pd.read_csv("data/ecowas_df_full_lag_zero.csv")

train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}
required_columnsv2 = {"gdp_o", "gdp_d", "dist"}

def baseline_loglinear(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016):
    '''
    The main function for creating our log-linear regression baseline
    
    Input:
        df = main pandas DataFrame to work with.
        target_variable = str of the column which we are inferring for. 
        columns = list of column elements that are included in the model. NOTE: Always include the basic Gravity columns, other we raise an error
        year_split = int of year where we split between train and test. Defaults to 2016
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    
    all_needed = columns + [target_variable]

    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = LinearRegression()
    model.fit(X_train, y_train)


    # Finally we can get some workable metrics from this regression:
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample R²:", r2)


    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                                cov_kwds={"groups": groups},)

    print(ols.summary())


    return {
        "model_name": target_variable,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }

    
def main(random_state:int = 42):
    '''
    
    '''

    np.random.seed(random_state)


c:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\bachelor_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# TO IMPLEMENT:

# SHAP summary for the variables in the ML models, to see which parameter has the most weight. 


legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic", "comlang_ethno", "comlang_off", "contig"}

conflict_groups = {
    "disorder":    [c for c in df.columns if c.startswith("disorder_")],
    "event":       [c for c in df.columns if c.startswith("event_")],
    "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
    "target":      [c for c in df.columns if c.startswith("target_")],
}

base_groups = list(conflict_groups.keys())

In [ ]:
def multilayer_perceptron(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016, set_random_state=42,
    hidden_layer=(100, 50),  alpha=0.0001, learning_rate_init=0.001):
    '''
    Function for applying multilayer perceptrons to the dataframe. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns = 
        year_split = 
        set_random_state = 
        hidden_layer = 
        alpha = 
        learning_rate_init =
    
    output:
        model = 
        dict containing various measures of fit.
    '''

    columns = columns.copy()

    # We do the standard checks for variables
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")
    
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")
    
    # Prepare the data frames
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()
    
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    # log the values for both train and test.
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Update the list to have the new log strings
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    
    # We encode the strings
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()), 
            ("mlp", MLPRegressor(
                hidden_layer_sizes=hidden_layer, 
                activation="relu",  
                solver="adam", 
                alpha=alpha,
                learning_rate_init=learning_rate_init,
                max_iter=1000, 
                random_state=set_random_state
            )),
        ]
    )

    model.fit(X_train, y_train)

    # SHAP is a little more complicated for Pipelines
    def predict_fn(X):
        '''
        Predicting on the dataframe, called by the KernelExplainer. 
        Please note, veeeery slow, so run on subsets of the data
        '''
        return model.predict(pd.DataFrame(X, columns=X_train.columns))
    
    background = shap.sample(X_train, 100) # Define a smaller set of X_train to run the code on. Otherwise it will take -forever-
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(X_train, silent=True)
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"MLP_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }


def xgboost_regression(df: pd.DataFrame, target_variable: str, columns: list, year_split: int = 2016, set_random_state: int = 42,
    n_estimators: int = 500, learning_rate: float = 0.05, max_depth: int = 6, subsample: float = 0.8, colsample_bytree: float = 0.8, min_child_weight: int = 1):
    '''
    Function for applying XGBoost to the dataframe. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns =
        year_split = 
        set_random_state = 
        n_estimators = 
        learning_rate = 
        max_depth = 
        subsample = 
        colsample_bytree = 
        min_child_weight = 
    
    output:
        model = 
        dict with various measures of fit.
    '''
    columns = columns.copy()

    # Same old quick validation checks
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")


    # Train/test split happens here
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()


    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables (for train/test)
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = pd.get_dummies(test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    # Then we can create our model
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        random_state=set_random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    # Final evaluation and scores - Here we also add the SHAP scores that will give us parameter weights!
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train) # These are SHAP values on all training data 
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value


    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"XGBoost_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }


def random_forest_regression(
    df: pd.DataFrame, target_variable: str, columns: list, year_split: int = 2016, set_random_state: int = 42,
    n_estimators: int = 500, max_depth: int | None = None, min_samples_leaf: int = 1, min_samples_split: int = 2, max_features: str | float = "sqrt"):
    '''
    Random Forest regression following the conventions and transformations of the baseline linear FE model. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns = 
        year_split = 
        set_random_state = 
        n_estimators = 
        max_depth = 
        min_samples_leaf = 
        min_samples_split = 
        max_features = 

    output:
        dict with various measures of fit.
    '''

    columns = columns.copy()

    # A few small validation checks, to make sure we have the right columns
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    # train/test split
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()


    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Replace raw gravity vars with log versions in columns list
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        # Align test columns to train (handles missing categories)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]



    # we create the random forest model, using the input variables as defined by the function call
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=set_random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # Final evaluation and scores - Here we also add the SHAP scores that will give us parameter weights!
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train) # These are SHAP values on all training data 
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"RandomForest_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }



def xgboost_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing xgboost performance with a varied combination of conflict inputs

    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = xgboost_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df



def random_forest_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing Random Forest performance with a varied combination of conflict inputs
    
    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = random_forest_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df

def mlp_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing Multilayer Perceptron performance with a varied combination of conflict inputs

    input:

    
    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    combos_to_test = []

    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))


    results = []
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = multilayer_perceptron(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)
    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — MLP {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [7]:
mlp_basic_model, dict_mlp_basic = multilayer_perceptron(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])


In [9]:
dict_mlp_basic

{'model_name': 'MLP_tradeflow_baci',
 'n_train': 2892,
 'n_test': 618,
 'rmse': 1.8374894432618476,
 'mae': 1.3215901496260327,
 'r2': 0.6998962713457881,
 'shap importance (grouped)': {'exporter_FE': np.float64(1.8821947884508807),
  'importer_FE': np.float64(0.6108497141858332),
  'other': {'contig': np.float64(0.5847365340510894),
   'comlang_off': np.float64(0.2591188516627256),
   'comlang_ethno': np.float64(0.047144361150442494),
   'log_gdp_o': np.float64(0.5316446929730052),
   'log_gdp_d': np.float64(1.0711239081112975),
   'log_distw': np.float64(0.37054175403761147)}}}

In [ ]:
# We need to run on a subset of files.

In [ ]:
rf_basic_model, dict_rf_basic = random_forest_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'], n_estimators=100)
rf_model, dict_rf = random_forest_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"], n_estimators=100)

mlp_basic_model, dict_mlp_basic = multilayer_perceptron(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
mlp_model, dict_mlp = multilayer_perceptron(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])

xgboost_basic_model, dict_xgboost_basic = xgboost_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
xgboost_model, dict_xgboost = xgboost_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])


In [11]:
dict_xgboost

{'model_name': 'XGBoost_tradeflow_baci',
 'n_train': 2892,
 'n_test': 618,
 'rmse': 2.1296505371963206,
 'mae': 1.5065732975592918,
 'r2': 0.5968762601227564,
 'shap importance (grouped)': {'exporter_FE': np.float32(1.1101698),
  'importer_FE': np.float32(0.46810687),
  'other': {'contig': np.float32(0.4491746),
   'comlang_off': np.float32(0.10808191),
   'comlang_ethno': np.float32(0.032321386),
   'disorder_Demonstrations': np.float32(0.06595184),
   'disorder_Political violence': np.float32(0.055114746),
   'disorder_Political violence; Demonstrations': np.float32(0.01419752),
   'disorder_Strategic developments': np.float32(0.05263044),
   'event_Battles': np.float32(0.036723703),
   'event_Explosions/Remote violence': np.float32(0.027408522),
   'event_Protests': np.float32(0.043209143),
   'event_Riots': np.float32(0.032676153),
   'event_Strategic developments': np.float32(0.0075428463),
   'event_Violence against civilians': np.float32(0.029581858),
   'perpetrator_Civilians':

In [ ]:



# --- 1) PPML conflict-ablation wrapper (mirrors find_best_conflict_combo1/2/3/4) ---
def find_best_conflict_combo5(df, target_variable, base_columns=None, year_split=2016):
    """
    PPML FE conflict ablation — same combo logic as combo1-4.
    

    
    """
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    cprefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in cprefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1) if all_conflict_cols else 0

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }
    
    base_groups = list(conflict_groups.keys())
    combos = []

    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos.append((" + ".join(combo), extra))
    all_cols = [c for g in base_groups for c in conflict_groups[g]]

    results = []
    for label, extra_cols in combos:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_ppml_fe_inline(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception:
            pass

    return pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)









# --- 2) CSV preprocessing: log1p on conflict columns ---
def preprocess_csv_for_grid(csv_path):
    df = pd.read_csv(csv_path)
    cprefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in cprefixes)]
    if "fatalities" in df.columns:
        conflict_cols.append("fatalities")
    for col in conflict_cols:
        df[col] = np.log1p(df[col])
    return df


# --- 3) Define the grid ---
ROOT = Path("C:/Users/mhm25/Desktop/ITU/6thSemester/bachelorproj/bachelor_2026/")

CSV_FILES = [
    # (filename, relative_path, source_type, lag, threshold)
    ("ecowas_df_full_lag_zero.csv",            "data/ecowas_df_full_lag_zero.csv",            "non-synthetic", 0, None),
    ("ecowas_df_full_lag_one.csv",             "data/ecowas_df_full_lag_one.csv",             "non-synthetic", 1, None),
    ("ecowas_df_full_lag_two.csv",             "data/ecowas_df_full_lag_two.csv",             "non-synthetic", 2, None),
    ("ecowas_df_synthetic_full_lag_zero_0.csv",   "data/synth/ecowas_df_synthetic_full_lag_zero_0.csv",   "synthetic", 0, 0),
    ("ecowas_df_synthetic_full_lag_zero_10.csv",  "data/synth/ecowas_df_synthetic_full_lag_zero_10.csv",  "synthetic", 0, 10),
    ("ecowas_df_synthetic_full_lag_zero_100.csv", "data/synth/ecowas_df_synthetic_full_lag_zero_100.csv", "synthetic", 0, 100),
    ("ecowas_df_synthetic_full_lag_one_0.csv",    "data/synth/ecowas_df_synthetic_full_lag_one_0.csv",    "synthetic", 1, 0),
    ("ecowas_df_synthetic_full_lag_one_10.csv",   "data/synth/ecowas_df_synthetic_full_lag_one_10.csv",   "synthetic", 1, 10),
    ("ecowas_df_synthetic_full_lag_one_100.csv",  "data/synth/ecowas_df_synthetic_full_lag_one_100.csv",  "synthetic", 1, 100),
    ("ecowas_df_synthetic_full_lag_two_0.csv",    "data/synth/ecowas_df_synthetic_full_lag_two_0.csv",    "synthetic", 2, 0),
    ("ecowas_df_synthetic_full_lag_two_10.csv",   "data/synth/ecowas_df_synthetic_full_lag_two_10.csv",   "synthetic", 2, 10),
    ("ecowas_df_synthetic_full_lag_two_100.csv",  "data/synth/ecowas_df_synthetic_full_lag_two_100.csv",  "synthetic", 2, 100),
]

TARGETS = ["tradeflow_baci", "tradeflow_imf_o", "tradeflow_imf_d",
           "tradeflow_comtrade_o", "tradeflow_comtrade_d", "combined_trade_baci"]

MODELS = {
    "XGBoost":       find_best_conflict_combo1,
    "Random Forest": find_best_conflict_combo2,
    "MLP":           find_best_conflict_combo3,
    "OLS Linear FE": find_best_conflict_combo4,
    "PPML FE":       find_best_conflict_combo5,
}

GRID_BASE_COLS = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

# --- 4) Output management with resumability ---
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
out_csv = results_dir / "robustness_grid.csv"

if out_csv.exists():
    grid_results = pd.read_csv(out_csv)
    completed = set(zip(grid_results["source_csv"], grid_results["model"], grid_results["target"]))
    print(f"Resuming run: {len(grid_results)} rows loaded, {len(completed)} (csv,model,target) cells already complete.")
else:
    grid_results = pd.DataFrame()
    completed = set()
    print("Starting fresh run — no prior results found.")

total_cells     = len(CSV_FILES) * len(MODELS) * len(TARGETS)
done_cells      = len(completed)
remaining_cells = total_cells - done_cells

print(f"\nGrid plan: {len(CSV_FILES)} CSVs × {len(MODELS)} models × {len(TARGETS)} targets = {total_cells} cells")
print(f"Done: {done_cells}.  Remaining: {remaining_cells}.\n")

# --- 5) Main loop ---
t_start    = time.time()
csv_cache  = {}   # avoid re-reading the same CSV multiple times within this run

for csv_name, rel_path, source_type, lag, threshold in CSV_FILES:
    full_path = ROOT / rel_path
    if not full_path.exists():
        print(f"⚠ MISSING: {rel_path}")
        continue

    # Lazy load with caching
    if csv_name not in csv_cache:
        try:
            csv_cache[csv_name] = preprocess_csv_for_grid(full_path)
        except Exception as e:
            print(f"⚠ {csv_name} failed to load: {e}")
            continue
    df_csv = csv_cache[csv_name]

    for model_name, combo_fn in MODELS.items():
        for target in TARGETS:
            if target not in df_csv.columns:
                continue
            cell_key = (csv_name, model_name, target)
            if cell_key in completed:
                continue

            t0 = time.time()
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    combo_df = combo_fn(df_csv, target, base_columns=GRID_BASE_COLS)

                if len(combo_df) == 0:
                    print(f"  ⊘ {csv_name} | {model_name} | {target}  — no successful combos")
                    continue

                combo_df = combo_df.copy()
                combo_df["source_csv"]  = csv_name
                combo_df["source_type"] = source_type
                combo_df["lag"]         = lag
                combo_df["threshold"]   = threshold
                combo_df["model"]       = model_name
                combo_df["target"]      = target

                grid_results = pd.concat([grid_results, combo_df], ignore_index=True)
                grid_results.to_csv(out_csv, index=False)
                completed.add(cell_key)
                done_cells += 1

                elapsed       = time.time() - t0
                total_elapsed = time.time() - t_start
                rate          = done_cells / total_elapsed if total_elapsed > 0 else 0
                eta_min       = (total_cells - done_cells) / rate / 60 if rate > 0 else 0

                print(f"  [{done_cells:>3}/{total_cells}]  "
                      f"{csv_name:<48} | {model_name:<14} | {target:<22} "
                      f"({elapsed:>5.1f}s)  ETA: {eta_min:>5.0f} min")

            except Exception as e:
                print(f"  ⚠ {csv_name} | {model_name} | {target}  FAILED: {type(e).__name__}: {str(e)[:80]}")

print(f"\n✓ DONE. Total runtime: {(time.time() - t_start) / 60:.1f} min.")
print(f"  Final result row count: {len(grid_results)}")
print(f"  Saved to: {out_csv}")

In [ ]:
# === Step 13b: Summary pivots over the robustness grid ===
# Run after the grid completes. Loads results/robustness_grid.csv and produces
# (a) best model per (csv, target), (b) does conflict help anywhere?,
# (c) gravity-only R² heat-map by csv × target × model.

grid = pd.read_csv(ROOT / "results" / "robustness_grid.csv")
print(f"Loaded {len(grid):,} rows from robustness_grid.csv")
print(f"Unique combos: {grid[['source_csv','model','target']].drop_duplicates().shape[0]}")

# Helper: identify gravity-only rows (model_name describes conflict combo, gravity-only is "none" — but our
# combo functions don't produce a "none" row. Gravity-only exists only as a separate result outside the combo
# wrapper. So we approximate gravity-only as "the row with the smallest set of conflict cols" — which is
# "fatalities" (single col) or "total_conflicts" (single col). We'll just use the *worst* combo per (csv, model, target)
# as a rough lower bound and the *best* combo as the upper bound. For a true gravity-only baseline we would need
# to call xgboost_regression / etc. directly on each CSV, which is outside this grid's scope.)

# (a) Best conflict combo per (csv, model, target) — by R²
best_per_cell = (
    grid.sort_values("r2", ascending=False)
        .groupby(["source_csv", "model", "target"], as_index=False)
        .first()
)
print("\n=== BEST CONFLICT COMBO per (CSV × model × target), by R² ===")
print(f"  {len(best_per_cell)} cells")

# (b) For each (csv, target), which model wins?
winner_per_target = (
    best_per_cell.sort_values("r2", ascending=False)
                 .groupby(["source_csv", "target"], as_index=False)
                 .first()
                 .rename(columns={"model": "winner_model", "r2": "winner_r2"})
                 [["source_csv", "target", "winner_model", "model_name", "winner_r2"]]
)
print("\n=== WINNER MODEL per (CSV × target) ===")
winner_counts = winner_per_target["winner_model"].value_counts()
print(winner_counts.to_string())

# (c) Mean R² by model, averaged across all (csv, target) cells (using best conflict combo per cell)
mean_r2_by_model = best_per_cell.groupby("model")["r2"].agg(["mean", "median", "std", "count"]).round(4)
print("\n=== Mean R² by model (across all CSV × target cells, using best conflict combo) ===")
print(mean_r2_by_model.sort_values("mean", ascending=False).to_string())

# (d) Mean R² by CSV (averaged across model × target × best combo)
mean_r2_by_csv = best_per_cell.groupby(["source_type", "lag", "threshold", "source_csv"])["r2"].agg(["mean", "count"]).round(4)
print("\n=== Mean R² by CSV (averaged across model × target) ===")
print(mean_r2_by_csv.sort_values("mean", ascending=False).to_string())

# (e) Mean R² by target (averaged across CSV × model × best combo)
mean_r2_by_target = best_per_cell.groupby("target")["r2"].agg(["mean", "median", "count"]).round(4)
print("\n=== Mean R² by target (averaged across CSV × model) ===")
print(mean_r2_by_target.sort_values("mean", ascending=False).to_string())

# (f) Best (csv, model, target, conflict_combo) overall
top10 = best_per_cell.nlargest(10, "r2")[["source_csv", "model", "target", "model_name", "rmse", "mae", "r2"]]
print("\n=== TOP 10 specifications across the entire grid ===")
print(top10.to_string(index=False))

# (g) Bottom 10 (worst-fitting cells)
bot10 = best_per_cell.nsmallest(10, "r2")[["source_csv", "model", "target", "model_name", "rmse", "mae", "r2"]]
print("\n=== BOTTOM 10 specifications ===")
print(bot10.to_string(index=False))

# Save the pivots
(ROOT / "results" / "robustness_winner_per_target.csv").write_text(winner_per_target.to_csv(index=False))
(ROOT / "results" / "robustness_best_per_cell.csv").write_text(best_per_cell.to_csv(index=False))
print(f"\nSummary tables saved to {ROOT / 'results'}/")